[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/DLCV-Lab-2026/blob/main/DLCV-05-Transfer-Learning-and-Evaluation/DLCV-05-Transfer-Learning-and-Evaluation.ipynb)

# Lab 05 — Transfer Learning, Augmentation and Evaluation

**Deep Learning for Computer Vision · AIT**

Labs 02–04 taught you to *define* models. This lab teaches you to *train one properly* —
which is where almost all real-world accuracy comes from, and where almost all student
projects lose it.

You will run the single most important experiment in applied deep learning:

> **The same architecture, on the same data, trained three ways.**
> From scratch · frozen backbone + linear head · full fine-tune.

Then we cover the things that separate a working model from a plausible-looking one:
augmentation, learning-rate discipline, honest evaluation, calibration, error analysis,
and **Grad-CAM** to check the model is looking at the right thing. We finish with
**DINOv2**, a self-supervised foundation model whose frozen features often beat anything
you can train yourself on a small dataset.

**Runs on:** Colab free T4, ~25 min total.

**Reading:** [Grad-CAM](https://arxiv.org/abs/1610.02391) ·
[Bag of Tricks](https://arxiv.org/abs/1812.01187) ·
[mixup](https://arxiv.org/abs/1710.09412) ·
[On Calibration of Modern NNs](https://arxiv.org/abs/1706.04599) ·
[DINOv2](https://arxiv.org/abs/2304.07193)

## 0. Setup

In [ ]:
!pip install -q timm scikit-learn

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt, time, copy

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16          # T4 = Turing: fp16 only, NOT bf16
torch.manual_seed(0); np.random.seed(0)
print("device:", device)

### The dataset: Flowers-102

We deliberately pick a dataset with **10 training images per class**. Small datasets are
the normal case in practice — nobody hands you ImageNet — and they are exactly where
transfer learning stops being an optimisation and becomes the difference between a
working model and a useless one.

If the Oxford mirror is down, the cell falls back to a small CIFAR-10 subset so the lab
still runs.

In [ ]:
IMG_SIZE = 224
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet statistics

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.14)), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

try:
    train_set = torchvision.datasets.Flowers102("./data", split="train", download=True, transform=train_tf)
    val_set   = torchvision.datasets.Flowers102("./data", split="val",   download=True, transform=eval_tf)
    NUM_CLASSES, DATASET = 102, "Flowers102"
except Exception as e:
    print("Flowers102 unavailable (%s); falling back to a CIFAR-10 subset." % type(e).__name__)
    full = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    val_full = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)
    idx = np.concatenate([np.where(np.array(full.targets) == c)[0][:100] for c in range(10)])
    train_set = torch.utils.data.Subset(full, idx)
    val_set = torch.utils.data.Subset(val_full, range(2000))
    NUM_CLASSES, DATASET = 10, "CIFAR10-subset"

train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("%s | %d train, %d val, %d classes" % (DATASET, len(train_set), len(val_set), NUM_CLASSES))
print("-> roughly %.0f training images per class" % (len(train_set) / NUM_CLASSES))

In [ ]:
def denorm(t):
    x = t.cpu().numpy().transpose(1, 2, 0) * np.array(STD) + np.array(MEAN)
    return np.clip(x, 0, 1)

xb, yb = next(iter(train_loader))
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for a, img, lab in zip(ax.flat, xb, yb):
    a.imshow(denorm(img)); a.set_title(int(lab), fontsize=9); a.axis("off")
plt.suptitle("%s - training samples (augmented)" % DATASET); plt.tight_layout(); plt.show()

---
# 1. The experiment that matters

Three ways to use a ResNet-18 on this dataset:

| Strategy | What trains | When to use |
|---|---|---|
| **A. From scratch** | everything, random init | you have >50k images, or your domain is nothing like ImageNet |
| **B. Linear probe** | only the final layer; backbone frozen | very little data, very little compute; also the standard way to *measure feature quality* |
| **C. Full fine-tune** | everything, starting from pretrained | the default. Almost always the right answer |

Note `weights="DEFAULT"` below. The old `pretrained=True` argument was deprecated in
torchvision 0.13 and the new API lets you name a specific weight set, which matters for
reproducibility.

In [ ]:
def make_model(strategy, num_classes=NUM_CLASSES):
    if strategy == "scratch":
        m = torchvision.models.resnet18(weights=None)
    else:
        m = torchvision.models.resnet18(weights="DEFAULT")     # was: pretrained=True

    if strategy == "linear_probe":
        for p in m.parameters():
            p.requires_grad = False                            # freeze the backbone

    m.fc = nn.Linear(m.fc.in_features, num_classes)            # new head, always trainable
    return m.to(device)


def trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

for s in ["scratch", "linear_probe", "finetune"]:
    print("%-14s trainable params: %6.2f M" % (s, trainable(make_model(s))))

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); logits, targets = [], []
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
        for x, y in loader:
            logits.append(model(x.to(device)).float().cpu())
            targets.append(y)
    return torch.cat(logits), torch.cat(targets)


def accuracy(model, loader):
    lg, y = predict(model, loader)
    return (lg.argmax(1) == y).float().mean().item()


def fit(model, epochs=6, lr=1e-3, label="run", mixup_alpha=0.0):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader), pct_start=0.25)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {"acc": [], "loss": []}
    t0 = time.time()

    for ep in range(epochs):
        model.train(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
                if mixup_alpha > 0:
                    lam = float(np.random.beta(mixup_alpha, mixup_alpha))
                    perm = torch.randperm(x.size(0), device=device)
                    x = lam * x + (1 - lam) * x[perm]
                    out = model(x)
                    loss = lam * crit(out, y) + (1 - lam) * crit(out, y[perm])
                else:
                    loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * y.size(0)
        acc = accuracy(model, val_loader)
        hist["loss"].append(running / len(train_set)); hist["acc"].append(acc)
        print("  [%s] ep %d/%d  loss %.3f  val acc %.4f" % (label, ep + 1, epochs, hist["loss"][-1], acc))

    print("  [%s] %.0fs | best %.4f" % (label, time.time() - t0, max(hist["acc"])))
    return hist

In [ ]:
results = {}
results["A. from scratch"]  = fit(make_model("scratch"),      epochs=6, lr=1e-3, label="scratch")
results["B. linear probe"]  = fit(make_model("linear_probe"), epochs=6, lr=1e-2, label="linear")
results["C. full finetune"] = fit(make_model("finetune"),     epochs=6, lr=3e-4, label="finetune")

In [ ]:
plt.figure(figsize=(7, 4.5))
for k, h in results.items():
    plt.plot(range(1, len(h["acc"]) + 1), h["acc"], marker="o", label=k)
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.title(DATASET)
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

for k, h in results.items():
    print("%-20s best val acc: %.4f" % (k, max(h["acc"])))

### Read this result carefully

Three things should be true, and each teaches something different:

1. **Scratch is terrible.** With ~10 images per class there is simply not enough signal
   to learn edges, textures and parts from nothing. This is not a bug in your code.
2. **A frozen backbone with one trainable layer beats it comfortably.** The ImageNet
   features already encode most of what "a flower" looks like. You are only learning how
   to read them.
3. **Full fine-tuning wins**, because it can also *adapt* those features.

Notice too how different the useful learning rates are: `1e-2` for the linear probe,
`3e-4` for fine-tuning. Fine-tuning at `1e-2` would destroy the pretrained weights in the
first few steps — a failure mode known as **catastrophic forgetting**, and the most
common reason a student's fine-tune does worse than their linear probe.

---
# 2. Discriminative learning rates

If early layers hold generic features (edges, colour blobs) and late layers hold
task-specific ones, they should not move at the same speed. **Discriminative learning
rates** give each parameter group its own LR: small near the input, large near the head.

In [ ]:
def param_groups(model, base_lr=3e-4, decay=0.3):
    # Later stages get a larger LR; the stem barely moves.
    stages = [model.conv1, model.bn1, model.layer1, model.layer2,
              model.layer3, model.layer4, model.fc]
    groups, n = [], len(stages)
    for i, s in enumerate(stages):
        groups.append({"params": list(s.parameters()),
                       "lr": base_lr * (decay ** (n - 1 - i))})
    return groups


m = make_model("finetune")
for i, g in enumerate(param_groups(m)):
    print("group %d  lr = %.2e  (%d tensors)" % (i, g["lr"], len(g["params"])))

In [ ]:
def fit_discriminative(model, epochs=6, base_lr=1e-3, label="disc-lr"):
    groups = param_groups(model, base_lr=base_lr)
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in groups], epochs=epochs,
        steps_per_epoch=len(train_loader), pct_start=0.25)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {"acc": []}
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        hist["acc"].append(accuracy(model, val_loader))
        print("  [%s] ep %d/%d  val acc %.4f" % (label, ep + 1, epochs, hist["acc"][-1]))
    return hist


best_model = make_model("finetune")
results["D. disc. LR + mixup"] = fit_discriminative(best_model, epochs=6, base_lr=1e-3)

---
# 3. Honest evaluation

A single accuracy number hides almost everything you need to know. Four diagnostics that
take ten lines each and will save you from shipping a broken model.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

logits, targets = predict(best_model, val_loader)
preds = logits.argmax(1)
probs = logits.softmax(1)

print("top-1 accuracy: %.4f" % (preds == targets).float().mean().item())
top5 = logits.topk(5, dim=1).indices
print("top-5 accuracy: %.4f" % (top5 == targets[:, None]).any(1).float().mean().item())

In [ ]:
cm = confusion_matrix(targets.numpy(), preds.numpy())
plt.figure(figsize=(7, 6))
plt.imshow(np.log1p(cm), cmap="viridis")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("confusion matrix (log scale)"); plt.colorbar(); plt.tight_layout(); plt.show()

# Which classes are actually failing?
per_class = cm.diagonal() / np.maximum(cm.sum(1), 1)
worst = np.argsort(per_class)[:10]
print("worst 10 classes (recall):")
for c in worst:
    print("  class %3d : %.2f   (n=%d)" % (c, per_class[c], cm.sum(1)[c]))

### Calibration: is a 0.9 confidence actually 90% correct?

Modern networks are systematically **overconfident**. If your model says 0.99 and is right
only 80% of the time, any downstream decision threshold you set is wrong. **Expected
Calibration Error** measures this: bin predictions by confidence and compare average
confidence to average accuracy within each bin.

In [ ]:
def expected_calibration_error(probs, targets, n_bins=15):
    conf, pred = probs.max(1)
    correct = (pred == targets).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece, xs, accs, confs = 0.0, [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        a, c = correct[m].mean().item(), conf[m].mean().item()
        ece += (m.float().mean() * abs(a - c)).item()
        xs.append(((lo + hi) / 2).item()); accs.append(a); confs.append(c)
    return ece, xs, accs, confs


ece, xs, accs, confs = expected_calibration_error(probs, targets)
print("Expected Calibration Error: %.4f" % ece)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
plt.plot(confs, accs, "o-", label="model")
plt.xlabel("mean confidence"); plt.ylabel("accuracy")
plt.title("Reliability diagram (ECE = %.3f)" % ece)
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

Points **below** the diagonal mean overconfidence. The cheapest fix is **temperature
scaling**: divide the logits by a single scalar T tuned on a validation set. It changes
no predictions — accuracy is identical — but makes the probabilities honest.

In [ ]:
def fit_temperature(logits, targets, iters=200):
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=iters)
    crit = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        loss = crit(logits / log_T.exp(), targets)
        loss.backward()
        return loss

    opt.step(closure)
    return log_T.exp().item()


T_opt = fit_temperature(logits.clone(), targets)
ece_after, *_ = expected_calibration_error((logits / T_opt).softmax(1), targets)
print("optimal temperature : %.3f" % T_opt)
print("ECE before / after  : %.4f -> %.4f" % (ece, ece_after))
print("accuracy unchanged  : %.4f" % ((logits / T_opt).argmax(1) == targets).float().mean().item())

### Error analysis: look at what it gets wrong

The highest-value 5 minutes in any project. Sort by confidence among the mistakes: a
confidently-wrong example is usually either a mislabelled image or a real conceptual gap.

In [ ]:
wrong = (preds != targets).nonzero().flatten()
conf_wrong = probs[wrong].max(1).values
order = wrong[conf_wrong.argsort(descending=True)][:8]

raw_val = torchvision.datasets.Flowers102("./data", split="val", transform=eval_tf) \
          if DATASET == "Flowers102" else val_set

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, i in zip(ax.flat, order):
    img, _ = raw_val[int(i)]
    a.imshow(denorm(img)); a.axis("off")
    a.set_title("true %d / pred %d\nconf %.2f" % (targets[i], preds[i], probs[i].max()), fontsize=9)
plt.suptitle("Most confident mistakes"); plt.tight_layout(); plt.show()

---
# 4. Grad-CAM: is it looking at the right thing?

A model can be right for the wrong reason — the classic case is a "horse" detector that
actually fires on a watermark. **Grad-CAM** answers *which spatial locations drove this
prediction* by weighting the last conv feature map by the gradient of the target class
with respect to it:

1. Forward the image, pick the target class score.
2. Backprop to the last convolutional feature map A.
3. Global-average-pool the gradients to get one weight per channel.
4. Weighted sum over channels, ReLU, upsample.

It is about 25 lines and needs no extra library.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model, self.acts, self.grads = model, None, None
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):
        self.acts = o.detach()

    def _save_grad(self, m, gi, go):
        self.grads = go[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval(); self.model.zero_grad()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1)
        out.gather(1, class_idx[:, None]).sum().backward()

        weights = self.grads.mean(dim=(2, 3), keepdim=True)      # (B, C, 1, 1)
        cam = F.relu((weights * self.acts).sum(1, keepdim=True))  # (B, 1, h, w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam - cam.amin(dim=(2, 3), keepdim=True)
        cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)
        return cam[:, 0].cpu(), class_idx.cpu()

In [ ]:
cam_model = best_model
gradcam = GradCAM(cam_model, cam_model.layer4[-1])

xb, yb = next(iter(val_loader))
x = xb[:6].to(device)
cams, cls = gradcam(x)

fig, ax = plt.subplots(2, 6, figsize=(15, 5.5))
for j in range(6):
    ax[0, j].imshow(denorm(xb[j])); ax[0, j].axis("off")
    ax[0, j].set_title("true %d" % yb[j], fontsize=9)
    ax[1, j].imshow(denorm(xb[j]))
    ax[1, j].imshow(cams[j], cmap="jet", alpha=0.45)
    ax[1, j].axis("off"); ax[1, j].set_title("pred %d" % cls[j], fontsize=9)
plt.suptitle("Grad-CAM: what drove the prediction"); plt.tight_layout(); plt.show()

Compare this with the **attention rollout** you will build in Lab 06. Grad-CAM needs
gradients and a convolutional feature map; attention rollout reads the model's own
attention weights directly. Both answer "where is it looking", by very different routes.

---
# 5. DINOv2: features you did not have to train

Everything so far started from **supervised** ImageNet weights. The field has largely
moved to **self-supervised** pretraining: learn from images with *no labels at all*, by
asking the model to make different views of the same image agree
([DINO](https://arxiv.org/abs/2104.14294)) or to reconstruct masked patches
([MAE](https://arxiv.org/abs/2111.06377)).

[DINOv2](https://arxiv.org/abs/2304.07193) is the practical outcome. Its frozen features
are strong enough that a **k-nearest-neighbour classifier with no training at all** is
often competitive with a fine-tuned ResNet on small datasets.

This section costs no training time — it is pure inference.

In [ ]:
import timm

dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0)
dino = dino.to(device).eval()
print("DINOv2 ViT-S/14 | embed dim:", dino.num_features,
      "| params (M): %.1f" % (sum(p.numel() for p in dino.parameters()) / 1e6))


@torch.no_grad()
def extract_features(model, loader, size=518 // 2):
    feats, labels = [], []
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
        for x, y in loader:
            x = F.interpolate(x.to(device), size=size, mode="bilinear", align_corners=False)
            feats.append(model(x).float().cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)


ftr, ytr = extract_features(dino, train_loader)
fva, yva = extract_features(dino, val_loader)
print("features:", tuple(ftr.shape), tuple(fva.shape))

In [ ]:
# (a) k-NN classifier: literally zero training.
ftr_n = F.normalize(ftr, dim=1)
fva_n = F.normalize(fva, dim=1)
sim = fva_n @ ftr_n.T
knn_pred = ytr[sim.topk(1, dim=1).indices[:, 0]]
knn_acc = (knn_pred == yva).float().mean().item()
print("DINOv2 frozen + 1-NN        : %.4f" % knn_acc)

# (b) Linear probe on the frozen features: a few seconds on CPU-sized tensors.
probe = nn.Linear(ftr.shape[1], NUM_CLASSES).to(device)
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
ftr_d, ytr_d, fva_d, yva_d = ftr.to(device), ytr.to(device), fva.to(device), yva.to(device)
for step in range(400):
    opt.zero_grad()
    loss = F.cross_entropy(probe(ftr_d), ytr_d)
    loss.backward(); opt.step()
probe_acc = (probe(fva_d).argmax(1) == yva_d).float().mean().item()
print("DINOv2 frozen + linear probe: %.4f" % probe_acc)
print("your best fine-tuned ResNet : %.4f" % max(results["D. disc. LR + mixup"]["acc"]))

Sit with that number for a moment. A model that **never saw a label during pretraining**,
used **completely frozen**, with a classifier you trained in a few seconds, is
competitive with — often better than — a ResNet you fine-tuned end to end.

This is the practical lesson of the foundation-model era, and it should change your
default plan for a new project:

> **Start with frozen foundation features and a linear probe.** It takes minutes, needs
> almost no compute, and gives you a strong baseline. Only fine-tune when you have
> measured that the probe is not enough.

The same logic underpins Lab 11, where a frozen vision encoder is bolted onto a frozen
language model and only the small connector between them is trained.

In [ ]:
summary_rows = [(k, max(h["acc"])) for k, h in results.items()]
summary_rows += [("E. DINOv2 frozen + 1-NN", knn_acc), ("F. DINOv2 frozen + probe", probe_acc)]
print("%-28s %s" % ("strategy", "val accuracy"))
print("-" * 44)
for k, v in sorted(summary_rows, key=lambda r: r[1]):
    print("%-28s %.4f  %s" % (k, v, "#" * int(v * 40)))

---
# Exercises

**1. The data-size curve.** Rerun strategies A, B and C using 10%, 25%, 50% and 100% of
the training set. Plot accuracy against training-set size for all three. At what dataset
size does training from scratch become competitive? This single plot is the most useful
thing you can show a client who asks "how much data do we need?".

**2. Catastrophic forgetting, measured.** Fine-tune at learning rates 1e-5, 1e-4, 1e-3,
1e-2 and 1e-1. Plot final accuracy against LR. Explain the shape of the curve in terms of
how far the weights are allowed to move from their pretrained values.

**3. Augmentation ablation.** Turn off `RandomResizedCrop`, then `RandomHorizontalFlip`,
then `mixup`, one at a time. Which matters most on this dataset, and why does the answer
depend on the dataset? (Hint: would a horizontal flip be safe for a digit classifier?)

**4. Calibration under distribution shift.** Apply a corruption to the validation set
(Gaussian blur, or a brightness shift) and recompute accuracy **and** ECE. Which degrades
faster? What does that imply about trusting a model's confidence in deployment?

**5. Grad-CAM debugging.** Find three validation images the model gets wrong and produce
Grad-CAM for both the true class and the predicted class. For each, state whether the
model was looking at the wrong region, or at the right region and reasoning wrongly.

**6. Foundation-feature bake-off.** Repeat the Section 5 k-NN and linear probe with
CLIP's image encoder (`timm` model `vit_base_patch16_clip_224.openai`) and with a
supervised `vit_base_patch16_224`. Rank the three feature extractors. Does the
self-supervised model win, and does the answer change with the number of labels per class?

**7. Your own dataset.** Take the dataset you built in Lab 04 Exercise 5 and run the full
pipeline: three strategies, confusion matrix, calibration, error analysis, Grad-CAM.
Write a one-page report on what your model actually learned. This is the template for
your final project.